# Phase 4 -- Automation & ETL Pipeline

**Goal:** Automate the Phase 1-3 workflow into reusable Python scripts.

## What is ETL?

| Step      | Meaning                   | In this project      |
| --------- | ------------------------- | -------------------- |
| Extract   | Pull raw data             | Read ibm_hr_raw.csv  |
| Transform | Clean + engineer features | All Phase 1 steps    |
| Load      | Save processed data       | Write hr_cleaned.csv |

## Why automate?

When new employee data arrives each month, you run one command:

```bash
python pipeline.py
```

Instead of re-running 90 notebook cells manually.

## What we build:

1. `pipeline.py` -- full ETL script
2. `report_gen.py` -- auto-generates a text report
3. `validate_data()` -- 6 data quality checks


## Step 1 -- Write the ETL Pipeline Script

**What this cell does:** Writes `pipeline.py` to disk. Three functions: `extract()` reads raw CSV, `transform()` applies all Phase 1 steps, `load()` saves the cleaned output. Logging records every step with timestamps.


In [1]:
import os
os.makedirs('../scripts', exist_ok=True)
os.makedirs('../logs', exist_ok=True)

lines = [
    'import pandas as pd, numpy as np, logging, os',
    'from datetime import datetime',
    'os.makedirs("../logs", exist_ok=True)',
    'logging.basicConfig(',
    '    level=logging.INFO,',
    '    format="%(asctime)s  %(levelname)s  %(message)s",',
    '    handlers=[logging.FileHandler("../logs/pipeline.log"), logging.StreamHandler()]',
    ')',
    'log = logging.getLogger(__name__)',
    '',
    'def extract(filepath):',
    '    log.info("Extracting: " + filepath)',
    '    df = pd.read_csv(filepath)',
    '    log.info("Extracted: " + str(df.shape[0]) + " rows")',
    '    return df',
    '',
    'def transform(df):',
    '    log.info("Starting transform...")',
    '    df.drop(columns=["EmployeeCount","StandardHours","Over18"], inplace=True, errors="ignore")',
    '    df["Attrition"] = df["Attrition"].map({"Yes":1,"No":0})',
    '   df["Income_Zscore"] = stats.zscore(df["MonthlyIncome"]).round(3)',
    '    df["OverTime"]  = df["OverTime"].map({"Yes":1,"No":0})',
    '    df["Gender"]    = df["Gender"].map({"Male":1,"Female":0})',
    '    for col in ["Department","JobRole","MaritalStatus","BusinessTravel"]:',
    '        if col in df.columns: df[col] = df[col].str.strip().str.title()',
    '    df["IsHighRisk"]        = np.where((df["OverTime"]==1) & (df["JobSatisfaction"]<=2), 1, 0)',
    '    df["AnnualIncome"]      = df["MonthlyIncome"] * 12',
    '    df["SatisfactionScore"] = df[["JobSatisfaction","EnvironmentSatisfaction","RelationshipSatisfaction","WorkLifeBalance"]].mean(axis=1)',
    '    df["TenureGroup"]       = pd.cut(df["YearsAtCompany"], bins=[-1,2,5,10,40], labels=["0-2","3-5","6-10","10+"])',
    '    df["DeptAvgIncome"]     = df.groupby("Department")["MonthlyIncome"].transform("mean")',
    '    df["Below_Dept_Avg"]    = np.where(df["MonthlyIncome"] < df["DeptAvgIncome"], 1, 0)',
    '    log.info("Transform done: " + str(df.shape[0]) + " rows x " + str(df.shape[1]) + " cols")',
    '    return df',
    '',
    'def load(df, output_path):',
    '    os.makedirs(os.path.dirname(output_path), exist_ok=True)',
    '    df.to_csv(output_path, index=False)',
    '    log.info("Saved: " + output_path)',
    '',
    'def run_pipeline():',
    '    start = datetime.now()',
    '    log.info("Pipeline STARTED")',
    '    df_raw   = extract("../data/ibm_hr_raw.csv")',
    '    df_clean = transform(df_raw)',
    '    load(df_clean, "../data/ibm_hr_stats_ready.csv")',
    '    elapsed = (datetime.now() - start).seconds',
    '    log.info("Pipeline DONE in " + str(elapsed) + "s")',
    '    return df_clean',
    '',
    'if __name__ == "__main__":',
    '    df = run_pipeline()',
    '    print("Pipeline complete.")',
]
with open('../scripts/pipeline.py', 'w') as f:
    f.write(chr(10).join(lines))
print('Created: ../scripts/pipeline.py')
print('Run with: cd scripts && python pipeline.py')

Created: ../scripts/pipeline.py
Run with: cd scripts && python pipeline.py


**What was created:** A fully self-contained ETL script. Each function has one job. Logging writes every step to `logs/pipeline.log` with timestamps. Running `python pipeline.py` does everything Phase 1 does -- automatically.


## Step 2 -- Test the Pipeline

**What this cell does:** Runs the pipeline inside the notebook to confirm all 6 engineered features are created correctly.


In [6]:
import pandas as pd, numpy as np, os, logging
from datetime import datetime

exec(open('../scripts/pipeline.py').read())
df_result = run_pipeline()

print()
print('Output: ' + str(df_result.shape[0]) + ' rows x ' + str(df_result.shape[1]) + ' columns')
print()
print('Engineered features confirmed:')
for col in ['IsHighRisk','AnnualIncome','SatisfactionScore',
            'TenureGroup','DeptAvgIncome','Below_Dept_Avg']:
    if col in df_result.columns:
        print('  OK  ' + col)
print()
print('Attrition rate: ' + str(round(df_result['Attrition'].mean()*100, 2)) + '%  (should be 16.12%)')

2026-04-16 23:54:48,154  INFO  Pipeline STARTED
2026-04-16 23:54:48,155  INFO  Extracting: ../data/ibm_hr_raw.csv
2026-04-16 23:54:48,189  INFO  Extracted: 1470 rows
2026-04-16 23:54:48,190  INFO  Starting transform...
2026-04-16 23:54:48,413  INFO  Transform done: 1470 rows x 39 cols
2026-04-16 23:54:48,482  INFO  Saved: ../data/ibm_hr_stats_ready.csv
2026-04-16 23:54:48,484  INFO  Pipeline DONE in 0s
2026-04-16 23:54:48,487  INFO  Pipeline STARTED
2026-04-16 23:54:48,488  INFO  Extracting: ../data/ibm_hr_raw.csv
2026-04-16 23:54:48,499  INFO  Extracted: 1470 rows
2026-04-16 23:54:48,502  INFO  Starting transform...
2026-04-16 23:54:48,518  INFO  Transform done: 1470 rows x 39 cols
2026-04-16 23:54:48,544  INFO  Saved: ../data/ibm_hr_stats_ready.csv
2026-04-16 23:54:48,546  INFO  Pipeline DONE in 0s


Pipeline complete.

Output: 1470 rows x 39 columns

Engineered features confirmed:
  OK  IsHighRisk
  OK  AnnualIncome
  OK  SatisfactionScore
  OK  TenureGroup
  OK  DeptAvgIncome
  OK  Below_Dept_Avg

Attrition rate: 16.12%  (should be 16.12%)


**Result:** Pipeline runs without errors. All 6 features confirmed. Attrition rate preserved at 16.12%.


## Step 3 -- Auto-Generate a Report

**What this cell does:** Writes `report_gen.py` which reads the cleaned CSV and produces a formatted text report automatically. In a real company this could be emailed to HR leadership every Monday.


In [7]:
lines_r = [
    'import pandas as pd, os',
    'from datetime import datetime',
    '',
    'def generate_report(data_path, output_path):',
    '    df   = pd.read_csv(data_path)',
    '    stay = df[df["Attrition"]==0]',
    '    left = df[df["Attrition"]==1]',
    '    now  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")',
    '    attrition_pct = round(df["Attrition"].mean() * 100, 2)',
    '    income_gap    = int(stay["MonthlyIncome"].mean() - left["MonthlyIncome"].mean())',
    '    ot_yes        = round(df[df["OverTime"]==1]["Attrition"].mean() * 100, 1)',
    '    ot_no         = round(df[df["OverTime"]==0]["Attrition"].mean() * 100, 1)',
    '    top_roles     = df.groupby("JobRole")["Attrition"].mean().mul(100).round(1).sort_values(ascending=False).head(3)',
    '    parts = [',
    '        "IBM HR ATTRITION -- AUTOMATED REPORT",',
    '        "Generated: " + now,',
    '        "=====================================",',
    '        "OVERVIEW",',
    '        "  Total      : " + str(len(df)),',
    '        "  Left       : " + str(int(df["Attrition"].sum())) + " (" + str(attrition_pct) + "%)",',
    '        "INCOME GAP",',
    '        "  Stayed avg : " + str(int(stay["MonthlyIncome"].mean())),',
    '        "  Left avg   : " + str(int(left["MonthlyIncome"].mean())),',
    '        "  Gap        : " + str(income_gap) + " per month",',
    '        "OVERTIME RISK",',
    '        "  With OT    : " + str(ot_yes) + "%",',
    '        "  Without OT : " + str(ot_no) + "%",',
    '        "TOP 3 ROLES",',
    '        top_roles.to_string(),',
    '        "====================================="',
    '    ]',
    '    report = chr(10).join(parts)',
    '    os.makedirs(os.path.dirname(output_path), exist_ok=True)',
    '    with open(output_path, "w") as f: f.write(report)',
    '    print(report)',
    '',
    'if __name__ == "__main__":',
    '    generate_report("../data/hr_cleaned.csv", "../reports/attrition_report.txt")',
]
with open('../scripts/report_gen.py', 'w') as f:
    f.write(chr(10).join(lines_r))
print('Created: ../scripts/report_gen.py')
print()
exec(chr(10).join(lines_r))
generate_report('../data/hr_cleaned.csv', '../reports/attrition_report.txt')

Created: ../scripts/report_gen.py

IBM HR ATTRITION -- AUTOMATED REPORT
Generated: 2026-04-16 23:57:19
OVERVIEW
  Total      : 1470
  Left       : 237 (16.12%)
INCOME GAP
  Stayed avg : 6832
  Left avg   : 4787
  Gap        : 2045 per month
OVERTIME RISK
  With OT    : 30.5%
  Without OT : 10.4%
TOP 3 ROLES
JobRole
Sales Representative     39.8
Laboratory Technician    23.9
Human Resources          23.1
IBM HR ATTRITION -- AUTOMATED REPORT
Generated: 2026-04-16 23:57:19
OVERVIEW
  Total      : 1470
  Left       : 237 (16.12%)
INCOME GAP
  Stayed avg : 6832
  Left avg   : 4787
  Gap        : 2045 per month
OVERTIME RISK
  With OT    : 30.5%
  Without OT : 10.4%
TOP 3 ROLES
JobRole
Sales Representative     39.8
Laboratory Technician    23.9
Human Resources          23.1


**Report contains:** Attrition rate, income gap (Phase 1 Finding 2), overtime multiplier (Phase 1 Finding 1), and top 3 highest-risk roles -- all computed fresh from the latest data each run.


## Step 4 -- Data Validation

**What this cell does:** 6 automated quality checks that run every time new data arrives. Each catches a real-world data problem before it corrupts downstream analysis.

- Missing columns = source schema changed
- Null target = attrition not recorded
- Too few rows = upstream pipeline failed
- Duplicate IDs = join error in HR system
- Impossible ages = data entry error
- Negative income = encoding error


In [8]:
def validate_data(df):
    errors, warnings = [], []
    required = ['Age','Attrition','MonthlyIncome','OverTime','Department','JobRole']
    missing  = [c for c in required if c not in df.columns]
    if missing: errors.append('Missing columns: ' + str(missing))
    if 'Attrition' in df.columns:
        nulls = int(df['Attrition'].isnull().sum())
        if nulls > 0: errors.append('Attrition has ' + str(nulls) + ' nulls')
    if len(df) < 100: warnings.append('Only ' + str(len(df)) + ' rows')
    if 'EmployeeNumber' in df.columns:
        dupes = int(df['EmployeeNumber'].duplicated().sum())
        if dupes > 0: errors.append(str(dupes) + ' duplicate EmployeeNumbers')
    if 'Age' in df.columns:
        bad = int(df[(df['Age'] < 18) | (df['Age'] > 70)].shape[0])
        if bad > 0: warnings.append(str(bad) + ' employees with Age outside 18-70')
    if 'MonthlyIncome' in df.columns:
        neg = int((df['MonthlyIncome'] < 0).sum())
        if neg > 0: errors.append(str(neg) + ' employees with negative income')
    print('DATA VALIDATION REPORT')
    print('=' * 42)
    if not errors and not warnings:
        print('  All 6 checks passed -- data is clean')
        return True
    for w in warnings: print('  WARNING : ' + w)
    if errors:
        for e in errors: print('  ERROR   : ' + e)
        print('  Pipeline BLOCKED')
        return False
    print('  Passed with warnings')
    return True

import pandas as pd
df_test  = pd.read_csv('../data/hr_cleaned.csv')
is_valid = validate_data(df_test)
print()
print('Result: ' + ('PASSED' if is_valid else 'FAILED'))

DATA VALIDATION REPORT
  All 6 checks passed -- data is clean

Result: PASSED


**Result:** All 6 checks pass on clean IBM data. In production with live data feeds, any ERROR blocks the pipeline automatically.


## Phase 4 -- Summary

**What was built:**

- `scripts/pipeline.py` -- ETL: Extract, Transform (Phase 1 steps), Load
- `scripts/report_gen.py` -- auto-generates text report
- `validate_data()` -- 6 automated quality checks
- Logging -- timestamped audit trail

**Folder structure:**

```
hr-attrition-analysis/
    data/          ibm_hr_raw.csv, hr_cleaned.csv, model_results.csv
    scripts/       pipeline.py, report_gen.py
    logs/          pipeline.log
    reports/       attrition_report.txt
    notebooks/     01_ to 05_.ipynb
```

**Why this matters:** Most fresher candidates show only notebooks. A working ETL script shows you understand how analysis moves from exploration to production.
